# PIDNet global concepts and concept perturbation

Reproducible workflow for the Flood dataset using the unchanged `galip_PIDNet` branch of [L-CRP](https://github.com/kortukov/L-CRP/tree/galip_PIDNet).

This notebook generates global concepts for background (class 0) and flood (class 1), runs deletion and insertion perturbation on the four post-merge layers, and saves aggregate AOC/AUC plots in `results/`. Existing experiment and L-CRP source files are not modified.

## 1. Configuration

Start Jupyter with the `pcx-galip` kernel/environment. Register it once from a terminal with `conda activate pcx-galip`, then `python -m pip install ipykernel`, followed by `python -m ipykernel install --user --name pcx-galip --display-name 'Python (pcx-galip)'`. Select **Python (pcx-galip)** in Jupyter. Change only the values in this cell when using another checkout or dataset location.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import shutil

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'paper/12-supp/code').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate repository root containing paper/12-supp/code')

REPO_ROOT = find_repo_root()
CODE_ROOT = REPO_ROOT / 'paper/12-supp/code'
LCRP_PARENT = REPO_ROOT / 'vendor/galip_PIDNet'
LCRP_ROOT = LCRP_PARENT / 'LCRP'
FLOOD_DATA_ROOT = REPO_ROOT / 'flood_segmentation'
RESULTS_ROOT = REPO_ROOT / 'results'

LAYERS = ['dfm.conv_p.0', 'dfm.conv_i.0', 'final_layer.conv1', 'final_layer.conv2']
CLASS_IDS = [0, 1]  # background, flood
REL_INIT = 'logits'
NUM_SAMPLES = 100
BATCH_SIZE = 5
NUM_STEPS = 23
SEED = 10
MIN_MASK_AREA = 64

print('Repository:', REPO_ROOT)
print('Python:', sys.executable)
print('Dataset:', FLOOD_DATA_ROOT)
print('L-CRP:', LCRP_ROOT)

Repository: /home/heydari/paper/Segmentation-and-Object-detection-PCX
Python: /home/heydari/miniconda3/envs/pcx-galip/bin/python
Dataset: /home/heydari/paper/Segmentation-and-Object-detection-PCX/flood_segmentation
L-CRP: /home/heydari/paper/Segmentation-and-Object-detection-PCX/vendor/galip_PIDNet/LCRP


## 2. Environment and input validation

The PIDNet loader used by these scripts requires a CUDA-capable environment. The checks below fail early with a useful message.

In [2]:
import numpy as np
import torch

assert sys.version_info[:2] == (3, 9), f'Expected Python 3.9, found {sys.version}'
assert int(np.__version__.split('.')[0]) < 2, f'Expected NumPy < 2, found {np.__version__}'
assert LCRP_ROOT.is_dir(), (
    f'Missing {LCRP_ROOT}. Clone with: git clone --branch galip_PIDNet --single-branch '
    f'https://github.com/kortukov/L-CRP.git {LCRP_ROOT}'
)
assert (FLOOD_DATA_ROOT / 'RGB/val/JPEG').is_dir(), 'Missing Flood validation images'
assert (FLOOD_DATA_ROOT / 'annotations/val/JPEG').is_dir(), 'Missing Flood validation masks'
assert torch.cuda.is_available(), 'CUDA GPU is required by the unchanged PIDNet model loader'

env = os.environ.copy()
python_paths = [str(LCRP_PARENT), str(CODE_ROOT)]
if env.get('PYTHONPATH'):
    python_paths.append(env['PYTHONPATH'])
env['PYTHONPATH'] = os.pathsep.join(python_paths)
env['FLOOD_DATA_ROOT'] = str(FLOOD_DATA_ROOT)
env['MPLBACKEND'] = 'Agg'
env['MPLCONFIGDIR'] = str(REPO_ROOT / '.matplotlib-cache')
Path(env['MPLCONFIGDIR']).mkdir(exist_ok=True)

print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.get_device_name(0))

/home/heydari/miniconda3/envs/pcx-galip/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 1.26.4
PyTorch: 1.13.1+cu117
CUDA: NVIDIA GeForce RTX 4090


In [3]:
# Confirm that imports resolve to the requested L-CRP checkout.
check = subprocess.run(
    [sys.executable, '-c', 'import LCRP; print(next(iter(LCRP.__path__)))'],
    cwd=REPO_ROOT, env=env, text=True, capture_output=True, check=True
)
resolved_lcrp = Path(check.stdout.strip()).resolve()
print('Imported L-CRP:', resolved_lcrp)
assert resolved_lcrp == LCRP_ROOT.resolve(), 'Python imported a different LCRP checkout'

Imported L-CRP: /home/heydari/paper/Segmentation-and-Object-detection-PCX/vendor/galip_PIDNet/LCRP


## 3. Helper for unchanged experiment scripts

In [4]:
def run_experiment(script, *args):
    command = [sys.executable, str(script), *map(str, args)]
    print('\n$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=REPO_ROOT, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    output = []
    for line in process.stdout:
        print(line, end='', flush=True)
        output.append(line)
    return_code = process.wait()
    if return_code:
        tail = ''.join(output[-40:])
        raise RuntimeError(f'Experiment failed with exit code {return_code}. Last output:\n{tail}')

GLOBAL_SCRIPT = CODE_ROOT / 'experiments/global_class_concepts.py'
PERTURB_SCRIPT = CODE_ROOT / 'experiments/instance_perturbation.py'
assert GLOBAL_SCRIPT.is_file() and PERTURB_SCRIPT.is_file()

## 4. Generate global concepts

This runs once for background and once for flood. It may take substantial GPU time. Set `RUN_GLOBAL_CONCEPTS = False` to reuse validated existing files.

In [5]:
RUN_GLOBAL_CONCEPTS = True
FORCE_REGENERATE_CONCEPTS = False
generation_concept_dirs = [
    RESULTS_ROOT / 'global_class_concepts/flood/pidnet' / REL_INIT,
    RESULTS_ROOT / 'global_class_concepts/flood_onlyflood/pidnet' / REL_INIT,
]

if RUN_GLOBAL_CONCEPTS:
    for class_id in CLASS_IDS:
        class_is_complete = any(
            all((directory / f'{layer}_class_{class_id}.pth').is_file() for layer in LAYERS)
            for directory in generation_concept_dirs
        )
        if not FORCE_REGENERATE_CONCEPTS and class_is_complete:
            print(f'Skipping class {class_id}: required concepts already exist.')
            continue
        run_experiment(
            GLOBAL_SCRIPT,
            '--model_name', 'pidnet',
            '--dataset_name', 'flood',
            '--class_id', class_id,
            '--batch_size', BATCH_SIZE,
            '--rel_init', REL_INIT,
        )


$ /home/heydari/miniconda3/envs/pcx-galip/bin/python /home/heydari/paper/Segmentation-and-Object-detection-PCX/paper/12-supp/code/experiments/global_class_concepts.py --model_name pidnet --dataset_name flood --class_id 0 --batch_size 5 --rel_init logits
INIT flood
Loading checkpoint from: /home/heydari/paper/12-supp/code/models/flood_model.pt


DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 2738
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 1675
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 2207
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 2636
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 1282
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 1792
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 1347
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 2314
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 1785
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 1269
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngI

CalledProcessError: Command '['/home/heydari/miniconda3/envs/pcx-galip/bin/python', '/home/heydari/paper/Segmentation-and-Object-detection-PCX/paper/12-supp/code/experiments/global_class_concepts.py', '--model_name', 'pidnet', '--dataset_name', 'flood', '--class_id', '0', '--batch_size', '5', '--rel_init', 'logits']' returned non-zero exit status 1.

In [ ]:
concept_candidates = [
    RESULTS_ROOT / 'global_class_concepts/flood/pidnet' / REL_INIT,
    RESULTS_ROOT / 'global_class_concepts/flood_onlyflood/pidnet' / REL_INIT,
]
def required_in(directory):
    return [directory / f'{layer}_class_{class_id}.pth' for layer in LAYERS for class_id in CLASS_IDS]

concept_dir = next((directory for directory in concept_candidates if all(p.is_file() for p in required_in(directory))), None)
if concept_dir is None:
    details = []
    for directory in concept_candidates:
        details.extend(str(path) for path in required_in(directory) if not path.is_file())
    raise FileNotFoundError('No complete concept directory found. Missing candidates:\n' + '\n'.join(details))
required_concepts = required_in(concept_dir)
print(f'Validated {len(required_concepts)} concept files in {concept_dir}')

## 5. Expose the concept path expected by perturbation

The existing perturbation script reads `results/flood/pidnet/logits`. A directory symlink provides that path without changing Python source.

In [ ]:
expected_parent = RESULTS_ROOT / 'flood/pidnet'
expected_parent.mkdir(parents=True, exist_ok=True)
expected_concepts = expected_parent / REL_INIT

if expected_concepts.is_symlink():
    if expected_concepts.resolve() != concept_dir.resolve():
        expected_concepts.unlink()
elif expected_concepts.exists():
    raise FileExistsError(f'Refusing to replace non-symlink path: {expected_concepts}')

if not expected_concepts.exists():
    expected_concepts.symlink_to(concept_dir, target_is_directory=True)
print(expected_concepts, '->', expected_concepts.resolve())

## 6. Run deletion and insertion perturbation

With balanced sampling and 100 samples, each run selects 50 background and 50 flood targets. Set `RUN_PERTURBATION = False` to reuse existing result tensors.

In [ ]:
RUN_PERTURBATION = True

if RUN_PERTURBATION:
    for insertion in (False, True):
        mode = 'insertion' if insertion else 'deletion'
        for layer in LAYERS:
            print(f'\n=== {mode}: {layer} ===')
            run_experiment(
                PERTURB_SCRIPT,
                '--model_name', 'pidnet',
                '--dataset_name', 'flood',
                '--layer_name', layer,
                '--rel_init', REL_INIT,
                '--num_samples', NUM_SAMPLES,
                '--batch_size', BATCH_SIZE,
                '--num_steps', NUM_STEPS,
                '--insertion', insertion,
                '--balanced_sampling', True,
                '--min_mask_area', MIN_MASK_AREA,
                '--seed', SEED,
            )

In [ ]:
perturb_dir = RESULTS_ROOT / 'instance_perturbation/flood/pidnet' / REL_INIT / 'data'
required_results = []
for layer in LAYERS:
    required_results += [
        perturb_dir / f'instance_perturbation_{layer}.pth',
        perturb_dir / f'instance_perturbation_{layer}_insertion.pth',
    ]
missing = [path for path in required_results if not path.is_file()]
assert not missing, 'Missing perturbation files:\n' + '\n'.join(map(str, missing))
print(f'Validated {len(required_results)} perturbation tensors in {perturb_dir}')

## 7. Aggregate and plot AOC deletion and AUC insertion

The aggregation matches the trapezoidal calculation used by `plot_instance_perturbation.py`. Shaded regions show the standard error over sampled instances.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

METHODS = ['LRP-zplus', 'LRP-gamma', 'LRP-eps', 'GradCAM', 'Gradient', 'activation', 'random']
METHOD_LABELS = {
    'LRP-zplus': 'LRP-z+', 'LRP-gamma': 'LRP-gamma', 'LRP-eps': 'LRP-eps',
    'GradCAM': 'GradCAM', 'Gradient': 'Gradient', 'activation': 'activation', 'random': 'random',
}

def load_result(layer, insertion):
    suffix = '_insertion' if insertion else ''
    path = perturb_dir / f'instance_perturbation_{layer}{suffix}.pth'
    return torch.load(path, map_location='cpu', weights_only=False)

def per_sample_area(result, method, insertion):
    steps = np.asarray(result['steps'], dtype=float)
    curves = np.asarray(result[method], dtype=float)
    if insertion:
        widths = np.diff(steps / steps[-1])
        curves = curves - curves[0:1]
        return np.trapz(curves, dx=widths[:, None], axis=0)
    widths = np.diff(np.concatenate([[0.0], steps]) / steps[-1])
    curves = np.concatenate([np.zeros_like(curves[:1]), curves], axis=0)
    areas = -np.trapz(curves, dx=widths[:, None], axis=0)
    nonzero = areas[areas != 0]
    return nonzero if nonzero.size else areas

def plot_summary(insertion):
    mode = 'insertion' if insertion else 'deletion'
    metric = 'AUC' if insertion else 'AOC'
    loaded = [load_result(layer, insertion) for layer in LAYERS]
    fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
    summary = {}
    for method in METHODS:
        values = [per_sample_area(result, method, insertion) for result in loaded]
        means = np.asarray([value.mean() for value in values])
        sems = np.asarray([value.std() / np.sqrt(max(1, value.size)) for value in values])
        summary[method] = {'layer_values': means, 'mean': float(means.mean())}
        line, = ax.plot(LAYERS, means, 'o-', linewidth=2, markersize=4,
                        label=f'{METHOD_LABELS[method]} ({means.mean():.3f})')
        ax.fill_between(LAYERS, means - sems, means + sems, color=line.get_color(), alpha=0.18)
    ax.set_title(f'PIDNet flood (with background) post-merge (logits) - {metric} {mode}')
    ax.set_ylabel(f'{metric} {mode}')
    ax.set_xlabel('Post-merge layers')
    ax.tick_params(axis='x', rotation=22)
    ax.legend(loc='upper left')
    fig.tight_layout()
    stem = RESULTS_ROOT / f'concept_perturbation_pidnet_with_background_{mode}'
    fig.savefig(stem.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
    fig.savefig(stem.with_suffix('.png'), dpi=300, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    print('Saved:', stem.with_suffix('.pdf'))
    print('Saved:', stem.with_suffix('.png'))
    return summary

deletion_summary = plot_summary(insertion=False)
insertion_summary = plot_summary(insertion=True)

## 8. Numeric summary

In [ ]:
import pandas as pd

summary_table = pd.DataFrame({
    'method': METHODS,
    'AOC deletion': [deletion_summary[m]['mean'] for m in METHODS],
    'AUC insertion': [insertion_summary[m]['mean'] for m in METHODS],
})
summary_table.round(6)